# DEPTH — EXP-002 training (Kaggle, GPU T4 ×2)

Trains the next DEPTH detector on dataset **v2b** and packages each model for one-command onboarding.

* **EXP-002** — YOLO11s at **1024 px** on full frames + 2×2 tiles (GPU 0)
* **EXP-002s** — the same at **640 px** (GPU 1) — STUDY-13 showed resolution must be *chosen* on
  held-out recordings, not assumed; onboarding compares the two within a speed budget.

Each run saves every epoch and exports the checkpoint with the best **ghost-gear AP@0.5 on the
held-out validation recordings** (not ultralytics' default box-tightness fitness).

**Settings:** Accelerator **GPU T4 ×2** · Internet **On** · then either **Save Version → Save & Run All
(Commit)** (runs unattended), or run cells 1–5 in order in this session and keep the tab open. Cell 5
shows what is running, waits, and collects; it is safe to re-run at any time. With one GPU the two runs
go one after the other. Outputs: `EXP-002_complete.zip`, `EXP-002s_complete.zip` in the Output tab.

In [ ]:
# 1. environment — the repo + the EXACT ultralytics version the kit was tested with
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!rm -rf depth && git clone -q https://github.com/madhesh60/depth.git && cd depth && git log --oneline -1
%cd /kaggle/working/depth
!pip install -q ultralytics==8.4.157
import torch, ultralytics, cv2
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA GPUs", torch.cuda.device_count(), "| cv2", cv2.__version__)
assert ultralytics.__version__ == "8.4.157", "wrong ultralytics version"
assert torch.cuda.is_available(), "no GPU - set Accelerator to GPU T4 x2"

In [ ]:
# 2. find the uploaded dataset (the folder holding data.yaml + train/ val/ test/) — no manual path
import glob, os
hits = [p for p in glob.glob("/kaggle/input/**/data.yaml", recursive=True) if os.path.isdir(os.path.join(os.path.dirname(p), "train"))]
assert hits, "dataset not found - add the depth-v2b dataset to this notebook (Add Input)"
SRC = os.path.dirname(sorted(hits, key=len)[0])
for s in ("train", "val", "test", "test_official398", "test_xsonar"):
    d = os.path.join(SRC, s, "images")
    print(f"{s:>17}: {len(os.listdir(d)) if os.path.isdir(d) else 'missing'} images")
print("SRC =", SRC)
assert len(os.listdir(os.path.join(SRC, "train", "images"))) > 1500, "train split looks incomplete"

In [ ]:
# 3. tiled training set in /kaggle/working (val/test stay the untouched full frames)
!python DATASET/scripts/build_tiles.py --src "{SRC}" --out /kaggle/working/v2b_tiles
!cat /kaggle/working/v2b_tiles/data.yaml
!echo "train images: $(ls /kaggle/working/v2b_tiles/train/images | wc -l)"
!df -h /kaggle/working | tail -1

In [ ]:
# 4. launch both runs (parallel on 2 GPUs, else one after the other) — logs in /kaggle/working/logs
#    Run it ONCE: it refuses to start a second copy while a training is alive. Cell 5 shows the status.
import shlex, subprocess, torch
os.makedirs("/kaggle/working/logs", exist_ok=True)
DATA = "/kaggle/working/v2b_tiles/data.yaml"
RUNS = [  # name, imgsz, batch
    ("EXP-002", 1024, 8),
    ("EXP-002s", 640, 16),
]
def cmd(name, imgsz, batch, dev):
    return ["python", "-u", "src/detection/train.py", "--data", DATA, "--name", name, "--imgsz", str(imgsz),
            "--batch", str(batch), "--device", str(dev), "--workers", "2", "--cache", "none",
            "--epochs", "30", "--patience", "8", "--notes", f"Kaggle T4, {imgsz}px, tiles, ghost-AP50 selection"]
live = subprocess.run(["bash", "-lc", "ps -eo args="], capture_output=True, text=True).stdout
assert "src/detection/train.py" not in live, "a training is already running - do not start it twice; run cell 5"
# start_new_session: the trainings get their own process group, so Stop/Interrupt on a cell cannot kill them
if torch.cuda.device_count() >= 2:
    for dev, (name, imgsz, batch) in enumerate(RUNS):
        log = open(f"/kaggle/working/logs/{name}.log", "w")
        subprocess.Popen(cmd(name, imgsz, batch, dev), stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        print("started", name, "on GPU", dev)
else:
    chain = " ; ".join(f"{shlex.join(cmd(n, s, b, 0))} > /kaggle/working/logs/{n}.log 2>&1" for n, s, b in RUNS)
    subprocess.Popen(["bash", "-c", chain], start_new_session=True)
    print("one GPU: the runs go one after the other:", ", ".join(n for n, *_ in RUNS))
print("now run cell 5: it shows what is running and waits until the models are packaged")

In [ ]:
# ==== DEPTH EXP-002 - STATUS / WAIT / COLLECT ====================================================
# Run it in a NEW cell, any time, as often as you like. It only LOOKS: it never starts, restarts or
# stops a training.
#   1. shows what each training is doing right now - read from the live process list and the run
#      files, so it works after a cell error or a kernel restart
#   2. while anything is still running: waits and prints a fresh status every CHECK_EVERY_MIN minutes
#   3. when nothing runs any more: copies the model packages to /kaggle/working (the Output panel)
# While it waits: keep this tab open, and do NOT press Stop/Interrupt or Restart. A Jupyter interrupt
# goes to the kernel's whole process group, which includes trainings launched without their own session.
import csv, json, os, re, shutil, subprocess, sys, time

WORK = "/kaggle/working"
RUNS_DIR, LOGS = f"{WORK}/depth/runs", f"{WORK}/logs"
NAMES = ["EXP-002", "EXP-002s"]          # 1024 px and 640 px
EPOCHS = 30                               # --epochs of cell 4 (patience 8 may end a run earlier)
CHECK_EVERY_MIN = 5
QUIET_WARN_MIN = 30                       # alive but nothing written for this long -> "may be stuck"


def sh(cmd):
    try:
        return subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception:
        return ""


def processes():
    """({run name: [(pid, seconds alive)]} for live trainings, whether any train.py process is alive).
    Data-loader workers (children of a training) and launcher shells are not counted as trainings."""
    procs = []
    for line in sh("ps -eo pid=,ppid=,stat=,etimes=,args=").splitlines():
        f = line.split(None, 4)
        if len(f) == 5 and "src/detection/train.py" in f[4] and not f[2].startswith("Z"):
            procs.append(f)
    py = {f[0] for f in procs if os.path.basename(f[4].split()[0]).startswith("python")}
    runs = {}
    for pid, ppid, _, et, args in procs:
        tok = args.split()
        if pid not in py or ppid in py or "--name" not in tok[:-1]:
            continue
        runs.setdefault(tok[tok.index("--name") + 1], []).append((int(pid), int(et) if et.isdigit() else 0))
    return runs, bool(procs)


def rows(n):
    p = f"{RUNS_DIR}/{n}/results.csv"
    if not os.path.exists(p):
        return []
    with open(p) as f:
        return [{k.strip(): (v or "").strip() for k, v in r.items() if k} for r in csv.DictReader(f)]


def log_tail(n, k=4):
    p = f"{LOGS}/{n}.log"
    if not os.path.exists(p):
        return ["(no log file)"]
    with open(p, "rb") as f:
        f.seek(0, 2)
        f.seek(max(0, f.tell() - 16000))
        txt = f.read().decode("utf-8", "replace")
    txt = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", txt).replace("\r", "\n")
    return [l.strip() for l in txt.split("\n") if l.strip()][-k:] or ["(log is empty)"]


def quiet_seconds(n):
    """Seconds since the run last wrote anything (log, metrics, checkpoints, selection)."""
    paths = [f"{LOGS}/{n}.log", f"{RUNS_DIR}/{n}/results.csv"]
    for d in (f"{RUNS_DIR}/{n}/weights", f"{RUNS_DIR}/{n}/select"):
        if os.path.isdir(d):
            paths += [os.path.join(d, x) for x in os.listdir(d)]
    t = [os.path.getmtime(p) for p in paths if os.path.exists(p)]
    return time.time() - max(t) if t else None


def phase(n):
    """What a live run is doing, read from its files."""
    run = f"{RUNS_DIR}/{n}"
    if os.path.isdir(f"{run}/select"):
        w = f"{run}/weights"
        total = len([x for x in os.listdir(w) if x.endswith(".pt") and x != "best_depth.pt"]) if os.path.isdir(w) else 0
        return f"CHOOSING THE BEST CHECKPOINT (validating {len(os.listdir(f'{run}/select'))} of {total})"
    if os.path.exists(f"{run}/selection.csv") or os.path.exists(f"{run}/weights/best_depth.pt"):
        return "EXPORTING ONNX + PACKAGING (a few minutes)"
    return "TRAINING"


def gpus():
    out = sh("nvidia-smi --query-gpu=index,utilization.gpu,memory.used,memory.total --format=csv,noheader,nounits")
    parts = []
    for line in out.splitlines():
        try:
            i, u, m, t = [x.strip() for x in line.split(",")]
            parts.append(f"GPU{i} {u}% busy, {float(m) / 1024:.1f}/{float(t) / 1024:.0f} GB")
        except ValueError:
            pass
    return " | ".join(parts) or "GPU info n/a"


def status():
    runs, any_alive = processes()
    print(time.strftime("\n=== %H:%M:%S (Kaggle clock) ==="), "|", gpus())
    for n in NAMES:
        r, alive = rows(n), runs.get(n, [])
        if alive:
            state = phase(n)
        elif os.path.exists(f"{RUNS_DIR}/{n}_complete.zip"):
            state = "FINISHED - package ready"
        elif not r and not os.path.exists(f"{LOGS}/{n}.log"):
            state = "NOT STARTED YET (queued)" if any_alive else "NOT STARTED"
        else:
            state = "STOPPED BEFORE FINISHING"
        print(f"{n:>9}: {state}")
        if r:
            last = r[-1]
            best = max(float(x.get("metrics/mAP50(B)") or 0) for x in r)
            info = (f"           epoch {len(r)}/{EPOCHS} | val mAP50 {float(last.get('metrics/mAP50(B)') or 0):.3f}"
                    f" (best so far {best:.3f}) | val recall {float(last.get('metrics/recall(B)') or 0):.3f}")
            if state == "TRAINING":
                per = (float(last.get("time") or 0) or max(et for _, et in alive)) / len(r)
                left = max(0, EPOCHS - len(r)) * per / 60
                info += f" | {per / 60:.1f} min/epoch -> at most ~{left:.0f} min + ~15 min to choose/export"
            print(info)
        elif alive:
            print(f"           starting - no finished epoch yet ({max(et for _, et in alive) / 60:.0f} min since launch)")
        if len(alive) > 1:
            newest = [str(p) for p, _ in sorted(alive, key=lambda x: x[1])[:-1]]
            print(f"           !! {len(alive)} copies of {n} are running (cell 4 was run twice); they share one folder."
                  f" Stop the newest with:  !kill {' '.join(newest)}")
        if alive:
            q = quiet_seconds(n)
            if q is not None and q > QUIET_WARN_MIN * 60:
                print(f"           !! alive but nothing written for {q / 60:.0f} min - may be stuck (see GPU % above)")
            print("           log:", log_tail(n, 1)[0][:150])
        elif state == "STOPPED BEFORE FINISHING":
            print("           last log lines:")
            for l in log_tail(n, 8):
                print("             ", l[:170])
    sys.stdout.flush()
    return any_alive


alive = status()
while alive:
    print(f"   ... still running - next check in {CHECK_EVERY_MIN} min. Keep this tab open; do NOT press Stop.")
    sys.stdout.flush()
    time.sleep(CHECK_EVERY_MIN * 60)
    alive = status()

print("\n=== nothing is running any more - collecting the results ===")
ok = 0
for n in NAMES:
    z = f"{RUNS_DIR}/{n}_complete.zip"
    if not os.path.exists(z):
        print(f"!! {n}: NO package - it stopped before finishing. Last log lines:")
        for l in log_tail(n, 15):
            print("    ", l[:200])
        continue
    shutil.copy(z, f"{WORK}/{n}_complete.zip")
    mp = f"{RUNS_DIR}/{n}/model_meta.json"
    m = json.load(open(mp)) if os.path.exists(mp) else {}
    sel = m.get("selection") or {}
    pick, ub = sel.get("picked") or {}, sel.get("ultralytics_best") or {}
    print(f"OK {n}: {(m.get('best') or {}).get('epochs_run')} epochs in {m.get('train_minutes')} min | exported "
          f"{pick.get('checkpoint')} -> ghost_gear AP50 {pick.get('ghost_ap50')} on val (ultralytics' best.pt: "
          f"{ub.get('ghost_ap50')}) | {m.get('cv2_dnn_verified')}")
    ok += 1
if ok == len(NAMES):
    shutil.rmtree(f"{WORK}/v2b_tiles", ignore_errors=True)       # the tiled copy is not needed any more
print(sh(f"ls -la {WORK}/*_complete.zip 2>/dev/null") or "no packages in /kaggle/working")
if ok:
    print("\nDONE. Download each *_complete.zip: right-hand panel -> Output (/kaggle/working) -> ... -> Download.")

## Next — on your own machine

Download `EXP-002_complete.zip` and `EXP-002s_complete.zip` from the **Output** tab into the repo root, then:

```bash
python -m src.detection.onboard_model --zip EXP-002_complete.zip  --max-ms 700
python -m src.detection.onboard_model --zip EXP-002s_complete.zip --max-ms 700
```

Keep the model with the better **recall ceiling / recall promise on the validation recordings within
the speed budget** (never pick by test), then `DEPTH_MODEL=<name>`.